# Lesson 01 Lab — The Inference Service Bottleneck

**Puzzle:** Why can a fast single prompt still become an unreliable concurrent service?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

A model demo measures one request after warm-up. A service owns a queue, admission policy, cache budget, batching policy, and latency objective. The relevant question is not whether the model can generate text, but whether the serving layer can convert irregular arrivals into useful GPU work without sacrificing tail latency.


## 0. Predict before running

1. Predict the output-token throughput for four short prompts.
2. Name one conclusion that an offline batch cannot establish.
3. Choose the additional trace needed before setting an online SLO.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The lab uses the installed vLLM engine, a local Qwen2.5 checkpoint, four prompts, generated-token counts, elapsed wall time, and CUDA memory observations. It records one native batch instead of inferring service behavior from a framework name.

- Single-request latency is not a concurrency result.
- Scheduling creates opportunities; workload shape determines whether they exist.
- A service decision needs both useful-token throughput and a tail-latency gate.


## 2. Derive the mechanism

Autoregressive inference alternates a compute-heavy prompt pass with repeated one-token steps. Independent requests reach those phases at different times. A serving engine can schedule ready token steps together and manage their KV state, but no scheduler can remove model work or guarantee an SLO without an arrival model. Throughput, latency, and queueing therefore form separate evidence axes.

### Mechanism at a glance

```mermaid
flowchart LR
  R["irregular requests"] --> Q["waiting queue"]
  Q --> S["token scheduler"]
  S --> G["GPU model step"]
  G --> K["KV state"]
  G --> O["streamed tokens"]
  O --> E["TTFT + ITL + throughput evidence"]
```

### Walk it step by step

1. **Separate the phases.** Treat prompt processing and token generation as different workloads.
2. **Make requests schedulable.** Expose ready work to one engine rather than isolated model loops.
3. **Measure useful work.** Count prompt and generated tokens beside elapsed time and memory.
4. **Bound the claim.** Add online queueing evidence before promising a service objective.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 1
LESSON_TITLE = 'The Inference Service Bottleneck'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260813
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | one frozen Qwen2.5 model and four independent prompts |
| Candidate | vLLM native offline batching for those prompts |
| Held constant | model path, sampling, prompt set, maximum output, seed, and GPU |
| Measurements | requests, prompt/output tokens, elapsed seconds, output tokens/s, and memory |
| Evidence | `native-backend` |

**Experiment:** Run one real vLLM offline batch and retain engine version, token counts, wall time, and GPU memory.


## 5. Inspect the experiment code

The experiment loads the checkpoint once, generates all requests in one call, and reads token IDs from each RequestOutput. It deliberately reports batch elapsed time rather than inventing per-request TTFT from an offline API.

Do not execute until the code matches the frozen table.


In [2]:
prompts = ["Explain continuous batching in two sentences.", "List two causes of high TTFT.",
           "What does a KV cache store?", "Why retain raw benchmark samples?"]
llm = LLM(**base_engine_args(max_model_len=1024))
params = SamplingParams(temperature=0.0, max_tokens=24, seed=SEED)
torch.cuda.reset_peak_memory_stats(); started = time.perf_counter()
outputs = llm.generate(prompts, params, use_tqdm=False); elapsed = time.perf_counter() - started
records = [output_record(item) for item in outputs]; output_tokens = sum(x["output_tokens"] for x in records)
metrics = {"vllm_version": vllm.__version__, "requests": len(records),
           "prompt_tokens": sum(x["prompt_tokens"] for x in records), "output_tokens": output_tokens,
           "elapsed_s": elapsed, "output_tokens_s": output_tokens / elapsed,
           "peak_allocated_mib": torch.cuda.max_memory_allocated() / 2**20, "outputs": records}
analysis = (f"vLLM {vllm.__version__} completed {len(records)} requests and {output_tokens} output "
            f"tokens in {elapsed:.3f} s ({metrics['output_tokens_s']:.1f} output tokens/s). This is "
            "native offline execution, not online queue or network evidence.")


INFO 08-13 00:15:09 [api_utils.py:273] non-default args: {'tokenizer': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'bfloat16', 'seed': 20260813, 'max_model_len': 1024, 'gpu_memory_utilization': 0.45, 'max_num_seqs': 16, 'disable_log_stats': True, 'enforce_eager': True, 'model': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct'}


INFO 08-13 00:15:09 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 08-13 00:15:09 [model.py:1883] Using max model len 1024


INFO 08-13 00:15:09 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.


WARNING 08-13 00:15:09 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


...

(EngineCore pid=649216) 2026-08-13 00:15:22,413 - INFO - autotuner.py:2397 - flashinfer.jit: [Autotuner]: Loaded 0 configs from <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/d10e66a551b175d65f9f24bcac568452ca3ec2ddfb6e3ee96a3fcc8c0723996c/autotune_configs.json
(EngineCore pid=649216) 2026-08-13 00:15:22,413 - INFO - autotuner.py:829 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=649216) 2026-08-13 00:15:22,471 - INFO - autotuner.py:852 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore pid=649216) 2026-08-13 00:15:22,477 - INFO - autotuner.py:2269 - flashinfer.jit: [Autotuner]: Saved 0 configs to <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/d10e66a551b175d65f9f24bcac568452ca3ec2ddfb6e3ee96a3fcc8c0723996c/autotune_configs.json (0 new, 0 from previous config)


(EngineCore pid=649216) INFO 08-13 00:15:23 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


(EngineCore pid=649216) INFO 08-13 00:15:23 [core.py:355] init engine (profile, create kv cache, warmup model) took 2.73 s


(EngineCore pid=649216) WARNING 08-13 00:15:23 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=649216) WARNING 08-13 00:15:23 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=649216) INFO 08-13 00:15:23 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=649216) INFO 08-13 00:15:23 [vllm.py:1426] Cudagraph is disabled under eager mode
(EngineCore pid=649216) INFO 08-13 00:15:23 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant


INFO 08-13 00:15:24 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| vLLM version | 0.27.1 |
| Requests | 4 |
| Prompt tokens | 29 |
| Output tokens | 96 |
| Elapsed | 0.225945 |
| Output throughput | 424.9/s |
| Peak allocated | 0.000 MiB |


## 7. Explain the result

vLLM 0.27.1 completed 4 requests and 96 output tokens in 0.226 s (424.9 output tokens/s). This is native offline execution, not online queue or network evidence.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`native-backend`**. The named vLLM runtime executed on the recorded GPU/model/workload. The result does not transfer to another version, model, endpoint, or traffic distribution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 1, "title": 'The Inference Service Bottleneck', "environment": ENV,
    "evidence_label": 'native-backend', "metrics": metrics,
    "analysis": analysis, "conclusion": 'The native run proves that this vLLM build serves the frozen batch on the RTX 5090; it does not prove a production concurrency SLO.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 1,
  "title": "The Inference Service Bottleneck",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260813
  },
  "evidence_label": "native-backend",
  "metrics": {
    "vllm_version": "0.27.1",
    "requests": 4,
    "prompt_tokens": 29,
    "output_tokens": 96,
    "elapsed_s": 0.225945346057415,
    "output_tokens_s": 424.8815108393754,
    "peak_allocated_mib": 0.0,
    "outputs": [
      {
        "request_id": "0",
        "prompt_tokens": 8,
        "output_tokens": 24,
        "token_ids": [
          68967,
          84256,
          374,
          264,
          1882,
          304,
          892,
          44792,
          315,
          7236,
          525,
          30878,
          3694,
          311,
          264,
          37629,
          476,
          18

## 9. Make the bounded decision

> The native run proves that this vLLM build serves the frozen batch on the RTX 5090; it does not prove a production concurrency SLO.

**Acceptance/rollback:** Adopt vLLM as the measured serving candidate only after online arrival tests also satisfy TTFT, ITL, error-rate, and memory gates.

**Failure analysis:** Warm-up, compilation, prompt length, sampling, and model size can reverse the observed rate. An offline batch does not contain network, tokenizer, queue, or streaming latency.


## 10. Extend the evidence

Replay a timestamped production-like arrival trace against the OpenAI endpoint and compare p50/p95 TTFT and ITL with the same model revision.

The full boundary and references are in [`README.md`](README.md).
